In [9]:
import pandas as pd

# Load the dataset
df = pd.read_csv('malicious_phish.csv')
print(df.head())

                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement


In [10]:
import requests
import time
from urllib.parse import urlparse
from bs4 import BeautifulSoup

def add_scheme_if_missing(url):
    """Ensure the URL has a scheme (http or https)."""
    parsed_url = urlparse(url)
    if not parsed_url.scheme:
        return 'http://' + url
    return url

def extract_html_features(url):
    """Extract HTML features from a webpage, handling errors gracefully."""
    url = add_scheme_if_missing(url)
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(url, timeout=30, headers=headers)
        response.raise_for_status()  # Will raise an HTTPError for bad responses
        soup = BeautifulSoup(response.content, 'html.parser')
        text = soup.get_text()  # Get all text from the page
        words = text.split()  # Split the text into words
        word_count = len(words)
        distinct_word_count = len(set(words))
        avg_word_length = sum(len(word) for word in words) / word_count if word_count > 0 else 0
        iframe_count = len(soup.find_all('iframe'))  # Count the number of iframes
        doc_length = len(response.text)  # Length of the HTML document
        return [word_count, distinct_word_count, avg_word_length, iframe_count, doc_length]
    
    except requests.exceptions.RequestException as e:
        print(f"Error processing {url}: {e}")
        return [0, 0, 0, 0, 0]  # Return zeros if an error occurs
    
    except Exception as e:
        print(f"Unexpected error with {url}: {e}")
        return [0, 0, 0, 0, 0]  # Return zeros if any unexpected error occurs

# Function to handle retries for DNS and timeout issues
def fetch_with_retries(url, retries=3, delay=10):
    """Retry fetching a URL in case of timeouts or connection errors."""
    for attempt in range(retries):
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
            return requests.get(url, timeout=30, headers=headers)  # Increased timeout to 30 seconds
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
            print(f"Timeout/Connection error on attempt {attempt + 1} for {url}: {e}")
            time.sleep(delay)
        except requests.exceptions.RequestException as e:
            print(f"Request error on attempt {attempt + 1} for {url}: {e}")
            break
    return None

# Sample URLs to test
urls = [
    'http://mp3raid.com/music/krizz_kaliko.html',
    'http://buzzfil.net/m/show-art/ils-etaient-loin-de-s-imaginer-que-le-hibou-allait-faire-ceci-quand-ils-filmaient-2.html',
    'http://espn.go.com/nba/player/_/id/3457/brandon-rush'
]

# Extract HTML features for each URL
for url in urls:
    response = fetch_with_retries(url)
    if response:
        html_features = extract_html_features(url)
        print(f"HTML Features for {url}: {html_features}")
    else:
        print(f"Failed to fetch {url} after multiple attempts.")
    time.sleep(3)  # Longer delay between requests





Failed to fetch http://mp3raid.com/music/krizz_kaliko.html after multiple attempts.
Timeout/Connection error on attempt 1 for http://buzzfil.net/m/show-art/ils-etaient-loin-de-s-imaginer-que-le-hibou-allait-faire-ceci-quand-ils-filmaient-2.html: HTTPConnectionPool(host='buzzfil.net', port=80): Read timed out. (read timeout=30)
Timeout/Connection error on attempt 2 for http://buzzfil.net/m/show-art/ils-etaient-loin-de-s-imaginer-que-le-hibou-allait-faire-ceci-quand-ils-filmaient-2.html: HTTPConnectionPool(host='buzzfil.net', port=80): Read timed out. (read timeout=30)
Timeout/Connection error on attempt 3 for http://buzzfil.net/m/show-art/ils-etaient-loin-de-s-imaginer-que-le-hibou-allait-faire-ceci-quand-ils-filmaient-2.html: HTTPConnectionPool(host='buzzfil.net', port=80): Read timed out. (read timeout=30)
Failed to fetch http://buzzfil.net/m/show-art/ils-etaient-loin-de-s-imaginer-que-le-hibou-allait-faire-ceci-quand-ils-filmaient-2.html after multiple attempts.
HTML Features for htt

In [6]:
# Extracting features for all URLs
content_features = df['url'].apply(extract_html_features)

# Converting the list of features into DataFrame
content_features_df = pd.DataFrame(content_features.tolist(), columns=[
    'word_count', 'distinct_word_count', 'avg_word_length', 'iframe_count', 'doc_length'
])



Error processing http://mp3raid.com/music/krizz_kaliko.html: 404 Client Error: Not Found for url: http://mp3raid.com/music/krizz_kaliko.html
Error processing http://buzzfil.net/m/show-art/ils-etaient-loin-de-s-imaginer-que-le-hibou-allait-faire-ceci-quand-ils-filmaient-2.html: HTTPConnectionPool(host='buzzfil.net', port=80): Read timed out. (read timeout=10)
Error processing http://espn.go.com/nba/player/_/id/3457/brandon-rush: 403 Client Error: Forbidden for url: http://espn.go.com/nba/player/_/id/3457/brandon-rush
Error processing http://www.pashminaonline.com/pure-pashminas: HTTPConnectionPool(host='www.pashminaonline.com', port=80): Max retries exceeded with url: /pure-pashminas (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x000001C4638054D0>, 'Connection to www.pashminaonline.com timed out. (connect timeout=10)'))
Error processing http://corporationwiki.com/Ohio/Columbus/frank-s-benson-P3333917.aspx: 403 Client Error: Forbidden for url: http://corpor

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Error processing http://almanac.logos.com/Denominations: HTTPConnectionPool(host='almanac.logos.com', port=80): Max retries exceeded with url: /Denominations (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000001C465761410>: Failed to resolve 'almanac.logos.com' ([Errno 11001] getaddrinfo failed)"))
Error processing http://gaming.wikia.com/wiki/Backbreaker_(video_game): 404 Client Error: Not Found for url: https://gamicus.fandom.com/wiki/Backbreaker_(video_game)
Error processing http://ceu.hu/: 403 Client Error: Forbidden for url: https://www.ceu.edu/
Error processing http://registry.weddingchannel.com/coupledir/20108/C/R314763282/JARED_CLOUGH_AND_TAJAI_RITCHIE.htm: HTTPConnectionPool(host='registry.weddingchannel.com', port=80): Max retries exceeded with url: /coupledir/20108/C/R314763282/JARED_CLOUGH_AND_TAJAI_RITCHIE.htm (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x000001C465882F90>: Failed to resolve 'registry.wedding

KeyboardInterrupt: 

In [ ]:
print(content_features_df.head())

In [ ]:
def extract_js_features(url):
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Extracting all <script> tags
        scripts = soup.find_all('script')
        js_code = " ".join([script.get_text() for script in scripts])

        # Counting suspicious JavaScript functions
        suspicious_functions = ['eval', 'unescape', 'escape', 'exec', 'document.write']
        js_feature_count = sum(js_code.count(func) for func in suspicious_functions)

        return js_feature_count
    except Exception as e:
        print(f"Error processing {url}: {e}")
        return 0

# Extracting JavaScript features for all URLs
js_features = df['url'].apply(extract_js_features)

# Adding JavaScript features to the DataFrame
content_features_df['js_feature_count'] = js_features


In [ ]:
print(content_features_df.head())